In [126]:
#changes from v3:
#migrate NG core calculation to new method...
#removed dict->list converter since calculate is performed in dictionaries
#v6: remove some test logs and try to complete the core deducer function
#v6b: hide some info log to support larger composition
#changes in v7:
#change to logic of deducer
#going to incorporate copilot help to improve the core deducer in v8

#20231121 ver outputs no error but the loop isn't working
#20231122 ver2 (output 7d 7e) fix some logical error, still errors not solved. Not it loops 9 times until it reach the breakpoint without enumerating new structures

In [127]:
#1127 fix the logic
#add the fuc function into the code, it's finished
#-
#enumerate through the terminal epitope list while taking arms number into account
#units left can have theoretically infinite times to add epitopes until bo more possibilities
#how about if the extra sia is getting depleted ur to some extra stupid reason?
#2 arms 4H4N3S2Fas example ...
## arm 1 SLeAX -1H1N1F1S = 3H3N2S1F
## arm 2 SLeAX - 1H1N1F1S = 2H2N1s

# terminal enumeration ends
###
## (should have epitopes both can be loxated at extending arms
## arm undefined (or shall we restart the count)
### sLNC -1H1N1S  = 1H1N
### LacNAc / finish enumaration
######
#will this method perform better than assigning epitopes after confirming all conponents inside? 
#collaborator said it was a simple math question (topological one?)

#1128 should have all except Fuc into the loop

In [128]:
#read needed packages, should execute everytime

#GlycoMSParser main version (for demo): 0.4 <- where should I store this information?
#version 0.6b (expect the v0.8 will work and v1.0 can provide one fully-automated process pipeline)

#needed packages
import pandas as pd
import time
import ast
from time import strftime, localtime
#not used in this version
mainver = "0.6b3"
modified_date = str(20231214)
version = mainver + modified_date
#import re

In [129]:
#read glycotope settings if there is one
#in alpha version it's hand-written in below cell
try:
    theoglycotopedict = pd.read_csv("glycotope_deducer.csv")
except:
    theoglycotopedict = {}

In [130]:
#this cell explain how glycotope header works
print("Glycotope header doc")
#explanations: name: name, used units: no. of units used for calculation, if we can deduce from structure, not from the
#monosaccharide units, then this part will become depreciated. enzyme list: 0 for absence, 1 for presence, "?" for unknown
#ion list = list of ions supporting this epitope, 
#groups: stem = N-glycan core, only used when other starting cores are not found, core: N-glycan cores (1 exclusive for each search)
#ext: extension possible, ter: terminal epitope, no longer extensible.
#type1: based on type1 lacnac, type2: based on type2 lacnac



#碎片資訊, 酵素資訊
#580 -> 存在的話4種結構都當可能並且輸出


def calcmassinepitope():
    print("developing")
#thinking if we can split epitopes into single component
#thinking if we can calculate fragments back from predicted structure (does that make sense in this pipeline?)
#should the ion be unique in all groups to avoid counted multiple times?



Glycotope header doc


In [131]:
#hand writing rules of glycotopes
glycotopeheader = ["Name of glycotope feature", "used units", "enzyme list and presence 0,1 or ?", "ion list", "groups"]


#this is defining epitope part
#TODO: change interger of gene expression to string, or do sth treat unknown expressions
#TODO: Need re-index for alias? homologs or different gene refer to same glycotope in different species
#TODO: calculate fragment automatically, not manual input
#######################glycotope list#######################
NG_core= ["NG core", {"H":3, "N":2}, {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "unknown enzyme": '?'}, [1234.56, 888.88], "stem"]
Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"MGAT3": 1}, [1234.56, 888.88], "core"]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {"MGAT4":1 ,"MGAT5":1}, [1234.56, 888.88], "core"]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {"B4GALT1":1, "B4TALT2~7": 1}, [187.0965, 209.1227], "ext"] #re?
corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter"]
LacNAc1 = ["Type 1 LacNAc", {"H":1, "N":1}, {"B3GALT2":1, "B3GALT5":1}, [464], "ext, type1"]
LacNAc2 = ["Type 2 LacNAc", {"H":1, "N":1}, {"B4GALT1~7":1}, [464, 432], "ext, type2"]
H_type1 = ["a2Fuc-Type1 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]
H_type2 = ["a2Fuc-Type2 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type2"]
LeA = ["Lewis A antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type1"]#b3-Gal type1 w a4-Fuc
LeX = ["Lewis X antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type2"]#b4-Gal type2 w a3-Fuc
LeB = ["Lewis B antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]#b3-Gal type1 w a4-Fuc, a2-Fuc at Gal
LeY = ["Lewis Y antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type2"]#b4-Gal type2 w a3-Fuc, a2-Fuc at Gal
Neu5Ac = ["Neu5Ac sialic acid", {"S": 1}, {"CMAS": 1}, [376],  "ext, ter"]
Neu5Gc = ["Neu5Gc sialic acid", {"G": 1}, {"CMAH": 1}, [406], "ext, ter"]
KDN = ["KDN sialic acid", {"KDN": 1}, {"CMP-KDN synthetase": 1}, [335], "ter"]
Sia3_LNAc1 = ["a2-3-Sialylated Type 1 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type1"]
Sia6_LNAc1 = ["a2-6-Sialylated Type 1 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type1"]
Sia3_LNAc2 = ["a2-3-Sialylated Type 2 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type2"]
Sia6_LNAc2 = ["a2-6-Sialylated Type 2 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type2"]
zf_GalExtension = ["a1-4 Galx2 + GlcNAc", {"H":2, "N":1}, {"unknown": "?"}, [668], "ext"]
zf_specific_epitope_5Ac = ["a1-4 Galx2 + GlcNAc + a2-3Sia + a1-3Fuc", {"F":1, "H":2, "N":1, "S":1}, {"unknown": "?"}, [1203], "ext"]
zf_specific_epitope_5Gc = ["a1-4 Galx2 + GlcNAc + a2-3Sia-Gc + a1-3Fuc", {"F":1, "H":2, "N":1, "G":1}, {"unknown": "?"}, [1233], "ext"]
LacDiNAc = ["LacDiNAc", {"N":2}, {}, [505], "ext, ter"]
Sia6_LacDiNAc = ["a2-6Sia_LacDiNAc", {"N":2}, {}, [505], "ext, ter"]
#######################glycotope list#######################
print("Add glycotopes into glycoenzymelist to include glycotopes defined above when needed.")

glycoenzymelist = [
    glycotopeheader,
    NG_core,
    Bisecting_NG_core,
    NG_elongationGal, 
    corefuc,
    LacNAc1,
    LacNAc2]

tmp = []

for glycoepitopes in glycoenzymelist:
    tmp.append(glycoepitopes)
    
    
#print(f"tmp is+ {tmp}")
c = pd.DataFrame(tmp)
c.columns = c.iloc[0]
c = c[1:]
#set first row as index column, copied from https://www.statology.org/pandas-set-first-row-as-header/
c


Add glycotopes into glycoenzymelist to include glycotopes defined above when needed.


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem
2,Bisecting NG core,"{'H': 3, 'N': 3}",{'MGAT3': 1},"[1234.56, 888.88]",core
3,First and elongation of Galactose at NG,{'H': 1},"{'B4GALT1': 1, 'B4TALT2~7': 1}","[187.0965, 209.1227]",ext
4,N-glycan core Fucosylation,{'F': 1},{'FUT8': 1},"[452.249, 697.3753]",ter
5,Type 1 LacNAc,"{'H': 1, 'N': 1}","{'B3GALT2': 1, 'B3GALT5': 1}",[464],"ext, type1"
6,Type 2 LacNAc,"{'H': 1, 'N': 1}",{'B4GALT1~7': 1},"[464, 432]","ext, type2"


In [132]:
#This part defines species information 

#TODO: make the gene expression become one scoring component after the algorithm is completed 

zebrafish_glycoT_brain = {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "absentenzyme": 0, "unknown": "?"}

print("This cell returns gene expression with value is 1 = known to be expressing the gene for defined species")
def findepitopefrom_glycoT(species):
    expression_list = []
    for key, value in species.items():
        if value == 1:
            expression_list.append(key)
    return expression_list
#those not expressed glycoT and unknown glycoT isn't returned in this version, consider if we need a way to recycle and use them

zfdemo = findepitopefrom_glycoT(zebrafish_glycoT_brain)
print(zfdemo)

This cell returns gene expression with value is 1 = known to be expressing the gene for defined species
['MGAT1', 'MGAT2', 'MAN2A1', 'MAN2A2']


In [133]:
print(f"This is testing cell")

print(len(c.index))
for i in range(1,len(c.index)+1):
    print(c["enzyme list and presence 0,1 or ?"][i])

c["enzyme list and presence 0,1 or ?"][1]
#print(c.loc[c["enzyme list and presence 0,1 or ?"]).isin(zfdemo)])

c.loc[1]

This is testing cell
6
{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}
{'MGAT3': 1}
{'B4GALT1': 1, 'B4TALT2~7': 1}
{'FUT8': 1}
{'B3GALT2': 1, 'B3GALT5': 1}
{'B4GALT1~7': 1}


0
Name of glycotope feature                                                      NG core
used units                                                            {'H': 3, 'N': 2}
enzyme list and presence 0,1 or ?    {'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...
ion list                                                             [1234.56, 888.88]
groups                                                                            stem
Name: 1, dtype: object

In [134]:
#write upper one into function
def findpossibleepitope(epitopes, species): 
    foundepitope = []
    index = 1  #need to adjust to correct indexed dataframe
    epitopelist = epitopes
    for i in epitopelist["enzyme list and presence 0,1 or ?"]:
        elist =[] #re-create to iterate through all rows and keep intact structure
        for key, value in i.items():
            if value == 1:
                elist.append(key)
        s = list(set(elist) & set(species))
        if s != []:
            foundepitope.append(epitopelist.loc[index])
            #should be appending the whole row
        else:
            print(f"There is no enzyme for this epitope {epitopelist.loc[index][0]}")
        index +=1

    foundepitopes = pd.DataFrame(foundepitope)
    return foundepitopes

findpossibleepitope(c, zfdemo)

deduceinput = findpossibleepitope(c, zfdemo)

There is no enzyme for this epitope Bisecting NG core
There is no enzyme for this epitope First and elongation of Galactose at NG 
There is no enzyme for this epitope N-glycan core Fucosylation
There is no enzyme for this epitope Type 1 LacNAc
There is no enzyme for this epitope Type 2 LacNAc
There is no enzyme for this epitope Bisecting NG core
There is no enzyme for this epitope First and elongation of Galactose at NG 
There is no enzyme for this epitope N-glycan core Fucosylation
There is no enzyme for this epitope Type 1 LacNAc
There is no enzyme for this epitope Type 2 LacNAc


/var/folders/lf/fb845wms1pjd3n_kz4w9yl_w0000gn/T/ipykernel_4414/1378633553.py:16: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"There is no enzyme for this epitope {epitopelist.loc[index][0]}")


In [135]:
print("this cells includes demo epitopes that don't check enzyme and peaklist")
#assigning core structure priority: largest core -> smallest core -> stem
#if core assignment successful -> try to extend based on possible "ext" epitopes
#if there's sugar unit left when no ext epitopes is possible, try to add "terminal" epitopes and ends of all values at 0
#if the enumeration cannot reach zero, then it means this ISNOT a valid structure
#DO NOT break the enumeration loop since we may get multiple assignment possible

#this step is HIGHLY POTENTIAL to get optimized by any of AI method since they do this kind of task better than human

#core structure
#arm number in the last
print("remove upper first LacNAc in the core ---- it should be extension part")

Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"Bisecting enzyme": 1}, [], "core", 2]
Biante_NG_core= ["Bi-antennary N-glycan core", {"H":5, "N":4}, {}, [], "core", 2]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {}, [], "core", 3]
Tetraante_NG_core = ["Tetra-antennary N-glycan core", {"H":7, "N":6}, {}, [], "core", 4]
Hybrid_NG_core = ["Bi-antennary hybrid N-glycan core", {"H":6, "N":3}, {}, [], "core", 1]

#stem structure, not included in the first demo
NG_core= ["NG core", {"H":3, "N":2}, {}, [], "stem"]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {}, [], "ext"] #re?

#higher priority to add core fucosylation if the enzyme is there
corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter, coreter"]


#calculate larger terminal structure to use out terminal units, from largest one
#maybe we can apply a sort method to do this based on sum of all key values in sugar unit dict

#calculate start from "each arm?"
#another idea here: try to put as much big units as possible for all arms
#for example, 3 arms, S x 3, F x 3, H x 5, N x 5 is left after core substraction

#example2: 2 arms, S x 2, F x 3, H x 2, N x 2

#calculate average S and F in each arm first
# ex1: S = 1, F = 1 while H, N = 1.66
# ex2: S = 1, F = 1.5 while H, N = 1
# when one value is larger than 1, try to assign a larger one epitope first.

#here we consider all same "composition" as one structure to validate our guess.
#in next exp we'll add enzyme list and peak list to compare and make sure which defines it's real possible structure... 


#if ratio N>H, add LDNA if enzyme permits it to do so
LacDiNAc = ["LacDiNAc", {"N":2}, {}, [505], "ext, ter"]

#for zebrafish case, we have certain structure need to be consider first
zf_specific_epitope_5Ac = ["a1-4 Galx2 + GlcNAc + a2-3Sia + a1-3Fuc", {"F":1, "H":2, "N":1, "S":1}, {"unknown": "?"}, [1203], "ext"]
zf_specific_epitope_5Gc = ["a1-4 Galx2 + GlcNAc + a2-3Sia-Gc + a1-3Fuc", {"F":1, "H":2, "N":1, "G":1}, {"unknown": "?"}, [1233], "ext"]


#2 Fucose
LeBY = ["Lewis BY antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]#b3-Gal type1 w a4-Fuc, a2-Fuc at Gal

#S+Fucose
sLeAX = ["Sialyl-Lewis AX antigen", {"F":1, "H":1, "N":1, "S":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]

#1 Fucose
Htype = ["Htype1 2 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]
LeAX = ["Lewis AX antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, type1"]#b3-Gal type1 w a4-Fuc

#1 Sialic acid
Sia_LNAc = ["Sialylated LNAc", {"H":1, "N":1, "S":1}, {}, [376.1966, 580, 792, 825], "ter, type1"]
SiaG_LNAc = ["SialylatedGc LNAc", {"H":1, "N":1, "G":1}, {}, [406.2072, 610, 822, 855], "ter, type1"]
KDN_LNAc = ["SialylatedKDN LNAc", {"H":1, "N":1, "KDN":1}, {}, [335, 539, 751, 784], "ter, type1"]


zf_GalExtension = ["a1-4 Galx2 + GlcNAc", {"H":2, "N":1}, {"unknown": "?"}, [668], "ext"]
#Sia6_LacDiNAc = ["a2-6Sia_LacDiNAc", {"N":2}, {}, [505], "ext, ter"]


#general extension if there is more extending units available
LacNAc = ["LacNAc", {"H":1, "N":1}, {"B3GALT2":1, "B3GALT5":1}, [464.2490], "ext, type1"]

#orphan units adds in the end
Neu5Ac = ["Neu5Ac sialic acid", {"S": 1}, {"CMAS": 1}, [376.1966],  "ter"]
Neu5Gc = ["Neu5Gc sialic acid", {"G": 1}, {"CMAH": 1}, [406.2072], "ter"]
KDN = ["KDN sialic acid", {"KDN": 1}, {"CMP-KDN synthetase": 1}, [335], "ter"]
#alphaGal = ["a1-4 Galx2 + GlcNAc orphan count", {"H":1}, {"unknown": "?"}, [668], "ter"]
#5Ac and 5Gc can be added on any arms with Sia at terminal

###extra when core arms = 0
GlcNAc = ["GlcNAc attached on core", {"N": 1}, {}, [260.1492],  "corrupt"]
LacNAc1 = ["LacNAc attached on core", {"N": 1, "H": 1}, {}, [464.2490],  "corrupt"]

cor_epitopes = [
    LacNAc1,
    GlcNAc,
]
#remove coreFuc and use another function to test it
demo_epitopes = [
    Bisecting_NG_core, 
    Biante_NG_core,
    Triante_NG_core,
    Tetraante_NG_core,
    Hybrid_NG_core,
    #corefuc,
    LeBY,
    sLeAX,
    Htype,
    LeAX,
    Sia_LNAc,
    SiaG_LNAc,
    KDN_LNAc,
    LacNAc,
    Neu5Ac,
    Neu5Gc,
    #alphaGal,
]
#only adds LacDiNAc when ion filter is ON
print(demo_epitopes)

#epitope 20231127
#groupA: LeBY, sLeAX, Sia_LNAc, SiaG_LNAc, KDN_LNAc, Htype, zf_GalExtension
#definition of groupA: terminal epitopes include basic LacNAc + other epitopes
#use 1 arm quota in each possible addition event
verb8_groupAepitopes = [
    LeBY, #F2H1N1
    sLeAX, #F1H1N1S1
    Sia_LNAc, #H1N1S1
    SiaG_LNAc, #H1N1G1
    KDN_LNAc, #H1N1K1
    LeAX, #F1H1N1
    zf_GalExtension, #H2N1
]
verb8_groupBeptiopes = [
    LeAX, #F1H1N1 which can be entensive unit
    LacNAc,
    LacDiNAc,
    Neu5Ac,
    Neu5Gc,
    KDN]

this cells includes demo epitopes that don't check enzyme and peaklist
remove upper first LacNAc in the core ---- it should be extension part
[['Bisecting NG core', {'H': 3, 'N': 3}, {'Bisecting enzyme': 1}, [], 'core', 2], ['Bi-antennary N-glycan core', {'H': 5, 'N': 4}, {}, [], 'core', 2], ['Tri-antennary N-glycan core', {'H': 6, 'N': 5}, {}, [], 'core', 3], ['Tetra-antennary N-glycan core', {'H': 7, 'N': 6}, {}, [], 'core', 4], ['Bi-antennary hybrid N-glycan core', {'H': 6, 'N': 3}, {}, [], 'core', 1], ['Lewis BY antigen', {'F': 2, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Htype1 2 LNAc', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, type1'], ['Sialylated LNAc', {'H': 1, 'N': 1, 'S': 1}, {}, [376.1966, 580, 792, 825], 'ter, type1'], ['Sialyla

In [136]:
import re

#finding and defining epitopes into groups
#should convert them into classes, methods to simplify this part of code...
print("testing cores")
coreelements = []
for i in demo_epitopes:
    if ("core" in i):
        coreelements.append(i)
        print(i)
        
extelements = []
print("testing extentives")
for i in demo_epitopes:
    if re.search("ext", i[4]):
        extelements.append(i)
        print(i)
        
        
terelements = []
print("testing terminal elements")
for i in demo_epitopes:
    if re.search("ter", i[4]):
        terelements.append(i)
        print(i)

#print(f"Core elements are {coreelements}")
#print(f"Extensive elements are {extelements}")
#print(f"Terminal elements are {terelements}")
print(f"compare same class w/ larger unit count should be iterated first.")
iter1 = []
orig = 0
for rank in coreelements:
    r = sum(rank[1].values())
    iter1.append((orig, r))
    orig+=1
    #print(iter1)
#sorting https://stackoverflow.com/questions/3121979/how-to-sort-a-list-tuple-of-lists-tuples-by-the-element-at-a-given-index
sorted_by_second = sorted(iter1, key=lambda tup: tup[1], reverse=True)
print(sorted_by_second)
sorted_coreelements = []
print("sorted list")
for sss in sorted_by_second:
    #print(sss[0])   #the original position
    sorted_coreelements.append(coreelements[sss[0]])
    print(coreelements[sss[0]])


iter2 = []
orig1 = 0
for rank in terelements:
    r = sum(rank[1].values())
    iter2.append((orig1, r))
    orig1+=1
    #print(f"{rank} iter2")
#sorting https://stackoverflow.com/questions/3121979/how-to-sort-a-list-tuple-of-lists-tuples-by-the-element-at-a-given-index
sorted_by_second1 = sorted(iter2, key=lambda tup: tup[1], reverse=True)
print(sorted_by_second1)
sorted_terelements = []
print("sorted terminal list")
for sss in sorted_by_second1:
    #print(sss[0])   #the original position
    sorted_terelements.append(terelements[sss[0]])
    print(terelements[sss[0]])

    
iter3 = []
orig2 = 0
for rank in extelements:
    r = sum(rank[1].values())
    iter3.append((orig2, r))
    orig2+=1
    #print(f"{rank} iter2")
#sorting https://stackoverflow.com/questions/3121979/how-to-sort-a-list-tuple-of-lists-tuples-by-the-element-at-a-given-index
sorted_by_second2 = sorted(iter3, key=lambda tup: tup[1], reverse=True)
print(sorted_by_second2)
sorted_extelements = []
print("sorted extensive list")
for sss in sorted_by_second2:
    #print(sss[0])   #the original position
    sorted_extelements.append(extelements[sss[0]])
    print(extelements[sss[0]])
    

testing cores
['Bisecting NG core', {'H': 3, 'N': 3}, {'Bisecting enzyme': 1}, [], 'core', 2]
['Bi-antennary N-glycan core', {'H': 5, 'N': 4}, {}, [], 'core', 2]
['Tri-antennary N-glycan core', {'H': 6, 'N': 5}, {}, [], 'core', 3]
['Tetra-antennary N-glycan core', {'H': 7, 'N': 6}, {}, [], 'core', 4]
['Bi-antennary hybrid N-glycan core', {'H': 6, 'N': 3}, {}, [], 'core', 1]
testing extentives
['Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, type1']
['LacNAc', {'H': 1, 'N': 1}, {'B3GALT2': 1, 'B3GALT5': 1}, [464.249], 'ext, type1']
testing terminal elements
['Lewis BY antigen', {'F': 2, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1']
['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1']
['Htype1 2 LNAc', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1']
['Sialylated LNAc', {'H': 1, 'N': 1, 'S': 1}, {}, [376.1966, 580, 792, 825], 'ter, type1']
['SialylatedGc LNAc

In [137]:
#another NG arms set w/o core N2H3
Bisecting_NG_arms= ["Bisecting NG core", {"H":2, "N":3}, {"Bisecting enzyme": 1}, [], "core", 2]
Biante_NG_arms= ["Bi-antennary N-glycan core", {"H":2, "N":2}, {}, [], "core", 2]
Triante_NG_arms= ["Tri-antennary N-glycan core", {"H":3, "N":3}, {}, [], "core", 3]
Tetraante_NG_arms = ["Tetra-antennary N-glycan core", {"H":4, "N":4}, {}, [], "core", 4]
Hybrid_NG_arms = ["Bi-antennary hybrid N-glycan core", {"H":7, "N":3}, {}, [], "core", 1]

corearms = [Tetraante_NG_arms, Triante_NG_arms, Biante_NG_arms, Bisecting_NG_arms, Hybrid_NG_arms]
#corearms[-1] = [4, 3, 2, 2, 1]

In [138]:
def predcomplsttodict(complist):
    predcomp = {}
    if len(complist) >= 5:
        predcomp["F"] = complist[0]
        predcomp["H"] = complist[1]
        predcomp["N"] = complist[2]
        predcomp["S"] = complist[3]
        predcomp["G"] = complist[4]
        if len(complist) == 6:
            predcomp["KDN"] = complist[5]
    print(f"predicted composition dict is {predcomp}")
    return(predcomp)

def checkifpossiblededuce(invalflag): #for debugging
    if (invalflag is True):
        print("This structural enumeration is not possible")
    elif(invalflag is False):
        print("Applying calculations...")
    else:
        raise ValueError("you haven't check if the enumeration is possible")
        
def compactcompcalc(calccomp, enumerateepitope, looplevel=0): #separate the elumarating METHOD outside as a independent function
        invalflag = None #init
        remaining_unit = [] #flag
        remaining_log = [] #Logs
        print(f"iteration on {enumerateepitope[0]}, nested loop depth= {looplevel}")
        valdict = {units: calccomp[units] - enumerateepitope[1].get(units, 0) for units in calccomp}
        #print(f"current valdict in compactcompcalc {valdict}")
        for unittest in valdict.values():
            if unittest < 0:
                invalflag = True
                #print("caught negative value")
                remaining_unit.append(False)
            else:
                remaining_unit.append(True)
                pass
        if (False in remaining_unit) is False:  #why this statement works?
            invalflag = False
        #checkifpossiblededuce(invalflag)  #for debug
        if enumerateepitope[1].keys() - calccomp.keys() == {"KDN"}:
            invalflag = True
            print("Find out KDN that's not in calculating composition")
        if invalflag is False:
            #print("add to logs")
            remaining_log.append(enumerateepitope[0]) #need to combine
            return True, remaining_log, valdict #added valdict
        elif invalflag is True:
            #print("do nothing. Test next glycotope")
            return False, "", valdict
        else:
            raise ValueError("The core assignment is somhow not executed.")

def unittest(valdict):
    remaining_unit = []
    invalflag = None
    for unittest in valdict.values():
        if unittest < 0:
            invalflag = True
            #print("caught negative value")
            remaining_unit.append(False)
        else:
            remaining_unit.append(True)
            pass
    if invalflag:  
        return False 
    else:
        #print("no negative values found")
        return True

def remintest(valdict): #added, not applied yet
    remin_flag = []
    for remins in valdict.values():
        if remins > 1:
            remin_flag.append(True)
        else:
            remin_flag.append(False)
    return remin_flag

def testzero(dictinput):
    for unittest in dictinput.values():
        if (unittest != 0): #set it invalid and leave no logs recorded
            return False
    return True

In [139]:
def coreFuccheck(input1, ionval = False, enzymeval = False):
    useFuc = False
    if ionval:
        print("Check the ions from the spectra")
        #after it pass the test
        useFuc = True
    if enzymeval:
        print("Check if enzyme is present")
        #after it pass the test
        useFuc = True
    if useFuc:
        print("placeholder for enzyme and ion fragments check")
    #corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter, coreter"]
    coreFuccalc = {units: input1[units] - corefuc[1].get(units, 0) for units in input1}
    if testzero(coreFuccalc):
        print("composition reaches zero")
        return True
    elif not unittest(coreFuccalc):
        print("coreFuc is impossible")
        return False
    else:
        return coreFuccalc

def avgtest(inputdict): #modified 20231122, since any zero value will cause this function return False
    judge = []
    #for reminame, reminno in inputdict.items():  #use average unit to get correct calc answer
    for residueavgunits in inputdict.values():
        judge.append(residueavgunits.is_integer()) #append False when the average unit is not integer
        #if residueavgunits < 1:
        #    judge.append(False)
        #    #print(f"there is no integer unit left for adding on {armsforepitopes} arm(s)")
        #elif residueavgunits >= 1.0:
        #    #print(f"integer unit left, able for adding terminal structure first")
        #    judge.append(True)
    if False in judge:
        return False
    else:
        return True
    return "Error"

In [140]:

from copy import deepcopy

def NGcorearms(spectrapredcomp):
    corededuce = {'H': 3, 'N': 2}
    coreelements = {units: spectrapredcomp[units] - corededuce.get(units, 0) for units in spectrapredcomp} #for further calculation
    #try 4 arms
    biante = {'H': 2, 'N': 2}
    triante = {'H': 3, 'N': 3}
    tetraante = {'H': 4, 'N': 4}
    bisect = {'H': 2, 'N': 3}
    hybrid = {'H': 7, 'N': 3}
    tmplist = [biante,triante,tetraante,bisect,hybrid]
    possiblearms = []
    i = 0
    #print(f"after subtract corededuce = {coreelements}")
    for arms in tmplist:
        arm = {units: coreelements[units] - arms.get(units, 0) for units in coreelements}
        #print(f"i = {i} and current calc is {arm}")
        #print(f"unit test on arm :{unittest(arm)}")
        if unittest(arm):
            if i == 0: #"biante":
                possiblearms.append(2)
                print(f"now possible arms is biante")
            elif i == 1: #arms == "triante":
                possiblearms.append(3)
                print(f"now possible arms is triante")
            elif i == 2: #arms == "tetraante":
                possiblearms.append(4)
                print(f"now possible arms is tetraante")
            elif i == 3: #arms == "bisect":
                possiblearms.append(2)
                print(f"now possible arms is bisect") #check if glycoT (MGAT5)? is present
            elif i == 4: #arms == "hybrid":
                possiblearms.append(1)
                print(f"now possible arms is hybrid")
        else:
            possiblearms.append(-1) #0 -> -1 to avoid conflicts
        i+=1
    #print(possiblearms)
    if possiblearms is []:
        possiblearms.append(-2) #-2 means error
    else:
        pass
    #print(f"possible arms {possiblearms}")
    #arms = 0
    arms = max(possiblearms)
    #consider returning all arms to reduce complexity in deducer
    return arms, coreelements 

def loopovertest():
    print("builing...should use this to replace loopoverflag")

def deducer(specific_epitopes, spectrapredcomp, debug =False):
    #deal with imported data
    #COPY to another dict to avoid updating on original dict instance!!!
    calc_comp = spectrapredcomp.copy()
    #group_NGcore = sorted_coreelements  #first level -> now we're using anothoer method, don't import it
    group_terminal = sorted_terelements #second level
    group_extensive = sorted_extelements #final level
    if debug:
        debug_logs = []
        with open("enumlogs_v8b.txt", "a") as debuglogs:
            curtime = time.time()
            x_curtime = strftime('%Y-%m-%d %H:%M:%S', localtime(curtime))
            debuglogs.write("\n ---Debug mode on--- \n")
            debuglogs.write("[dev]importing grouped epitope definition: group_terminal, group_extensive \n")
            debuglogs.write("version 20231127 beta8 copilot supported \n")
            debuglogs.write(str(x_curtime))
            debuglogs.write(version)
        print(f"[debug] {x_curtime} Mode on and logs has been initiated: {debug_logs}")
    #how to calculate sugar unit changes: #cdict = {key: calc[key] - bdict.get(key, 0) for key in adict}
    #when the composition becomes zero -> export all stuff to complog
    #validation apply to complog and so the finalresult is deduced (in dev)
    complog = [] 
    compno = 0 #feature, to separate enumeration results. +1 when start adding outputs for enumeration.
    finalresults = []
    armsno, forcalcomp= NGcorearms(calc_comp)
    if debug:
        notif = "[debug] input arms number " + str(armsno) + ", original comp" + str(spectrapredcomp) + ", Composition for calculation (-H3N2) " + str(forcalcomp)
        print(notif)
        debug_logs.append(notif + "\n")
    ###core###
    for armdeduce in corearms:
        armscomp = [] #RESET when we start a new "arm" enumeration
        #get possible arm numbers first
        #arms= -1 means the composition is less than H5N4
        if armsno == -1: #means only NG core may be possible
            #print("confirm if the current dict left zero")
            if debug:
                print("Start corrupted NG searching")
                debug_logs.append("[debug] armsno = -1, start corrupted NG searching" + "\n")
            if unittest(forcalcomp): #check negative value first, and then start terminal addition
                complog = [] #newly added 
                loopoverflag = True
                compforloop = forcalcomp
                while loopoverflag: #loops until all units get zeroed or any of negative value appears
                    for leftovers in cor_epitopes:
                        compforloop1 = {units: compforloop[units] - leftovers[1].get(units, 0) for units in compforloop}
                        #fix in 20231102: should only update composition if it's possible to add epitopes
                        #no idea why previous version works...
                        if unittest(compforloop1):
                            compforloop.update(compforloop1)
                            armscomp.append(leftovers[0])
                            if debug: #add epitope log into debug logs...
                                popupmsg = "[epitope added]: " + str(leftovers[0])
                                print(popupmsg)
                                debug_logs.append(popupmsg+ "\n")
                            #continues calculation but break if all remaining units = 0 or negative numbers
                            if testzero(compforloop):
                                loopoverflag = False
                                #armscomp.append("exits with 0") 
                                compno +=1
                                compcount = "[enumeration no:]" + str(compno)
                                complog.append(str(compcount))
                                complog.append(armscomp) #add all caught epitopes into armscomp and then export to complog when test zero.
                                if debug:
                                    debug_logs.append(str(compcount) + "\n")
                                    debug_logs.append("The composition reaches zero. Composition log is ")
                                    debug_logs.append(str(complog) + "\n")
                                finalresults.append(complog) #newly added, to make corrupted NG core follows the export format
                                complog = [] #reset the complog
                                break
                        else:
                            if debug:
                                print(f"{compforloop1} on {leftovers[0]} causing enumeration ends")
                            loopoverflag = False
                            break
                if debug:
                    end_corrputedNG = "[debug] Finished deducing corrupted NG core structure"
                    debug_logs.append(end_corrputedNG + "\n")
                break
            #only enter this block when the composition not satisfies NG core (making forcalcomp has neg value) 
            else:
                if debug:
                    end_unsatisfiedcorNG = "[debug] The composition couldn't satisfy even a simplest NG core."
                    debug_logs.append(end_unsatisfiedcorNG + "\n")
                break
        elif armsno == 0:#only happens when enumeration from larger arm numbers get subtracted. Get a break in the end to jump out this enumeration loop...
            print("Reached the arms no = 0 after descending enumerates arms no. Break the enumeration loop")
            if debug:
                debug_logs.append("Arm number reaches 0 and the enumeration should be stopped" + "\n")
            break
        else:
            ###############################Normal enumeration#############################################
            #print("need to assign forcalcomp everytime in the beginning: get original composition and start dealing with ..")
            #Start enumeration from largest arms
            #Hybrid and maybe bisect type need unique armsno
            print(f"arms no: {armsno}, enumerates on {armdeduce[-1]}")
            if armsno < armdeduce[-1]: #armdeduce starts from 4, ends in 1
                if debug:
                    popupmsg = "skipping impossible arms enumeration current:" +  str(armsno)  + " vs ref: " +  str(armdeduce[-1])
                    debug_logs.append(popupmsg + "\n")
                continue
            elif armsno == armdeduce[-1]:
                #get the composition left fitting current arms no, for example biantennary takes H3N2+ H2N2 = H5N4
                valdict = {units: forcalcomp[units] - armdeduce[1].get(units, 0) for units in forcalcomp}
                if debug:
                    popupmsg = str(armsno) + "arms enumerating and now the valdict is " +  str(valdict)  + " with enumerating on " +  str(armdeduce[0])
                    debug_logs.append(popupmsg + "\n")
                    print(popupmsg)
                #test if VALDICT is valid for further calculation
                #check if the composition gets zero. If so, record and exit. Otherwise start terminal enumeration
                validflag = unittest(valdict)
                if validflag: #invalflag is False:
                    if testzero(valdict): #Stop enumeration if composition already reaches zero
                        compno +=1
                        compcount = "[enumeration no:]" + str(compno)
                        complog.append(str(compcount))
                        complog.append(armdeduce[0]) #armdeduce ->corededuce #log ->complog
                        complog.append("0") #exit with 0
                        finalresults.append(complog)
                        complog = [] #reset the complog
                        armsno -=1
                        if debug:
                            print(f"Reaches 0 when only fitting basic structure for NG arms {armdeduce[0]}")
                            debug_logs.append(compcount + "\n")
                            debug_logs.append(str(complog) + "\n")
                            debug_logs.append("Enumeration exits when fitting basic structures" + "\n")
                        continue
                        #do we need this?
                    else:
                        #Possible for composition enumeration
                        #complog.append(armdeduce[0])
                        #complog.append("core+arms+?")
                        ########################################################################################
                        #armsno = global one; armsforepitopes = local one for enumerating all epitopes
                        #try to add glycotopes from as much arms as possible. when not possible, -1 -> -2 arm
                        #assign temp composition storing space and isolate the calculating composition
                        armsforepitopes = deepcopy(armsno) #copy CURRENT arms number for enumeration (DO NOT modify original instance)
                        if debug:
                            terstart = "now the temp looping arms = " + str(armsforepitopes) + ". Start terminal enumeration."
                            print(terstart)
                            debug_logs.append(terstart + "\n")
                        print(f"[dev] Calculate avg units carrying on arms = {armsforepitopes}. Real arms = {armsno}")
                        while armsforepitopes > 0:
                            #print("Calculate avg first...")
                            #print("If all arms can afford epitopes then start looping")
                            tempcomp = []
                            tempcomp.append(armdeduce[0])
                            #print("v6 changing the logic here")
                            ff = forcalcomp.copy() #reset the composition to original - H3N2
                            #valdict1 = valdict.copy() #reset the composition to original - (core+arms)
                            #add copy (20231026) to prevent referring to same object
                            #reject_terloop = False #check if this flag-driven loop is compactible to  current code
                            #reject_extloop = False
                            #tmplog = [] #store terminal and extensive informatiom, reset to None in each "loop"
                            avgunit = {} #clear the avgunits
                            if debug:
                                print(f"reset avg {avgunit}, comp {ff} and local arms {armsforepitopes}, global {armsno}")
                                debugout = "after reset {}, composition {} and arms no local =  {}, global = {}".format(avgunit,ff,armsforepitopes,armsno)
                                debug_logs.append(debugout + "\n")
                            print("rebuiling logic here")
                            locater = 0 #to locate first epitope being added.
                            print("locater +1 when able to add. In next loop we don't need it, start from next one...")

                            #thinking.... now it is not working.... we should have 3 outputs for F1H5N4S1 but it only gives one back
                            #need quota for adding epitopes
                            gatekeeper = 0 #to prevent infinite loop
                            #dev use
                            locatermonitor = 0
                            #-1 quota when epitope is added
                            #end the loop when locater starts from last element of group_terminal
                            if debug:
                                locatercheck = "check condition entering locater loop..." + str(locater) + " vs length of terminal " + str(len(verb8_groupAepitopes))
                                debug_logs.append(locatercheck + "\n")
                            #start counting quota here (global one)
                            #quota = deepcopy(armsno) #real arms quota for coming iteration
                                


                            #consider a trigger when testzero happens
                            enum_node = False
                            # change it to true when testzero(comp)
                            if enum_node == True:
                                print("reset arms for enu, quota and start from 'next' epitope ")
                                #think that if the 
                            #reset after catch a testzero and start from next epitope....
                            while locater <= len(verb8_groupAepitopes):
                                #monitor and debug exception handler
                                #maybe the locater should be changed as well if a run shows no epitope can be added
                                locatermonitor +=1
                                if debug:
                                    debug_logs.append(f"[dev]now locater is {locater} and looped {locatermonitor} times" + "\n")
                                gatekeeper+=1
                                if gatekeeper > 9:
                                    if debug:
                                        debug_logs.append("[debug] gatekeeper reached 10, break the loop and check the code" + "\n")
                                    break
                                if locatermonitor > 9:
                                    if debug:
                                        debug_logs.append("[debug] locatermonitor reached 10, break the loop and check the code" + "\n")
                                    break
                                #variable preset (reset)
                                quota = deepcopy(armsno) #real arms quota for coming iteration
                                #in the loop while armsforepitopes > 0:
                                #do avg calc here
                                #reminflag = []
                                for reminame, reminno in ff.items(): 
                                    avgunit[reminame] = (reminno/armsforepitopes)
                                #check if the average sugar units are possible for addition of epitopes
                                if debug:
                                    #add avgunits info into debug logs
                                    debug_logs.append("avgunit is " + str(avgunit) + "\n") #does it work? by copilot


                                #check if the average sugar units are in integer so we can add epitopes in BATCH (same units on multiple arms)
                                if avgtest(avgunit):
                                    if debug:
                                        avgtestpassed = "avg test passed, start trying to add epitopes on " + str(armsforepitopes) + " arms" #should have info added into complog
                                        debug_logs.append(avgtestpassed + "\n")
                                    #do the terminal epitope adding, start from 0 but when possible structure is added, locater will be changed to skip those had been enumerated
                                    #do the quota test here...
                                    #when more unit left and quota is already 0 (such as 1 7 6 1 going to only 2 arm-> after core 1 4 4 1 -> sLeAX 0 3 3 0 -> LNAc -> 0 2 2 0 -> quota = 0 -> re-allocate?)
                                    testflag = [] #to avoid useless loop?
                                    while (quota > 0 and locater <= len(verb8_groupAepitopes)):
                                        print(f"[dev]quota is {quota} and locater is {locater}")
                                        if testflag == []:
                                            print("Unexpected error")
                                            if debug:
                                                debug_logs.append("Unexpected error or first run, testflag is empty" + "\n")
                                        elif True in testflag:
                                            if debug:
                                                debug_logs.append("testflag is True, continue adding epitopes" + "\n")
                                            testflag = [] #reset the testflag
                                        elif False in testflag:
                                            if debug:
                                                debug_logs.append("False in the testflag, no epitopes can be added, break the loop" + "\n")
                                            locater = len(verb8_groupAepitopes) #to force it leave the loop
                                            quota = 0 #to force it run the extensive loop  
                                        else:
                                            if debug:
                                                debug_logs.append("testflag exception, just let it go" + "\n")
                                        for i in verb8_groupAepitopes[locater:]: #group_terminal-> verb8
                                            #output debug info
                                            if debug:
                                                terminaldb = "[v8b groupA]Looping terminal start from " + str(i[0]) + ", avg unit = " + str(avgunit) + ", locater = " + str(locater) + " arms" + str(armsforepitopes)  + '\n'
                                                debug_logs.append(terminaldb)
                                                debug_logs.append("quota left is " + str(quota) + "in group A epitope" +"\n")
                                            #continue checking if the composition didnt reach zero
                                            if not testzero(avgunit):
                                                #do the terminal enumeration
                                                a, b, secvaldict = compactcompcalc(avgunit, i, 2)
                                                #a will be True if the epitope can be added 
                                                #b isnt being used currently, consider remove it
                                                if a:
                                                    testflag.append(True)
                                                    #c = "".join(b) #convert epitope log list to str
                                                    #update the avgunit soon after the epitope is added, in this loop
                                                    avgunit.update(secvaldict) 
                                                    #######
                                                    if debug:
                                                        debug_logs.append(f"[dev]avgunit get updated to {avgunit}")
                                                    quota-=1
                                                    complog.append(i[0]) #if the error happens, change to str(i[0])
                                                    if testzero(secvaldict):#when the composition reaches zero
                                                        #set the quota to 0 to avoid it enters the b loop
                                                        quota = 0 
                                                        #update the locater based on first epitope being added
                                                        for kkkkk in verb8_groupAepitopes[locater:]:
                                                            if str(kkkkk[0]) == str(complog[0]):
                                                                iint = verb8_groupAepitopes.index(kkkkk)
                                                                print(f"[dev]: index of {kkkkk[0]} is {type(iint)} {iint}")
                                                                skipiter = "[dev]: index of " + str(kkkkk[0]) + " is " + str(type(iint)) + str(iint) + '\n'
                                                                #dev, to tell
                                                                if locater >= iint:
                                                                    pass
                                                                else:
                                                                    try:
                                                                        locater = int(iint) +1
                                                                        #locater = int(iint) +1 #start from next we had found
                                                                        resetlocater = "[dev] reset locater to " + str(locater) + '\n'
                                                                        if debug:
                                                                            debug_logs.append("Set quota to 0 as well" + "\n")
                                                                            debug_logs.append(skipiter)
                                                                            debug_logs.append(resetlocater)
                                                                        #print(resetlocater)
                                                                    except:
                                                                        print("locater can't be assigned")
                                                                break

                                                        #print result and export complog to finalresult, reset complog for next enumeration
                                                        print("terminal units fits zero. Export to complog")
                                                        compno +=1
                                                        compcount = "[enumeration no:]" + str(compno)
                                                        complog.append(str(compcount) + "\n")
                                                        #add core information after finding correct index of epitope
                                                        complog.insert(0, armdeduce[0])
                                                        finalresults.append(complog)
                                                        quotaleft = "available arm quota remaining: " + str(quota) + "in terminal enumeration" +'\n'
                                                        if debug:
                                                            debug_logs.append(str(compcount) + "\n")
                                                            debug_logs.append(str(quotaleft) + "\n")
                                                            debug_logs.append("now the complog is" + "\n")
                                                            debug_logs.append(str(complog) + "\n")
                                                        complog = [] #reset the complog   
                                                        break #leave the for i loop?   
                                                else:
                                                    testflag.append(False)
                                                    #locater+=1 #to force it start from next epitope after one loop searching
                                                    continue
                                            else: #when the composition goes to zero
                                                if debug:
                                                    debug_logs.append("found zero in terminal looping" + "\n")
                                                break
                                            #otherwise try to add epitopes
                                        #decide if we should enter extensive loop by quota left
                                        locater +=1 #to force it start from next epitope after one loop searching
                                        if debug:
                                            debug_logs.append("Entering terminal enumeration and quota left is " + str(quota) + "\n")
                                    """
                                    if quota > 0:
                                        #try to add extensive epitopes
                                        extcount = 0 #debug preventing infinite loop
                                        while quota > 0:
                                            extcount +=1
                                            extbreak = [] #list

                                            #infinite loop prevention
                                            if len(extbreak) == len(group_extensive):
                                                print("[dev] all extensive epitopes are not possible to add")
                                                break
                                            elif extcount > 9:
                                                print("[dev] the extensive break is not working, force break")
                                                break
                                            ########################
                                            for j in group_extensive:
                                            #add extensive epitopes if the composition is not zero
                                                aa, bb, finalvaldic = compactcompcalc(avgunit, j, 3)
                                                if debug:
                                                    extendb = "Looping terminal start from " + str(j[0]) + ", avg unit = " + str(avgunit) + ", locater = " + str(locater) + '\n'
                                                    debug_logs.append(extendb)
                                                    debug_logs.append("quota left is " + str(quota) + "\n")
                                                if aa:
                                                    cc = "".join(bb)
                                                    avgunit.update(finalvaldic) #update the avgunit in this loop
                                                    print(f"[dev]avgunit get updated to {avgunit} in extensive loop")
                                                    quota-=1
                                                    complog.append(j[0]) #if the error happens, change to str(i[0])
                                                    if testzero(finalvaldic):#when the composition reaches zero
                                                        #ll = locater + 1
                                                        for kkkkk in group_terminal[locater:]:
                                                            if str(kkkkk[0]) == str(complog[0]):
                                                                iint = group_terminal.index(kkkkk)
                                                                print(f"[dev]: indx of {kkkkk[0]} is {type(iint)} {iint}")
                                                                skipiter = "[dev]: indx of " + str(kkkkk[0]) + " is " + str(type(iint)) + str(iint) + '\n'
                                                                #dev, to tell
                                                                if locater >= iint:
                                                                    pass
                                                                else:
                                                                    try:
                                                                        locater = int(iint) +1
                                                                        #locater = int(iint) +1 #start from next we had found
                                                                        resetlocater = "[dev] reset locater to " + str(locater) + '\n'
                                                                        if debug:
                                                                            debug_logs.append(skipiter)
                                                                            debug_logs.append(resetlocater)
                                                                        print(resetlocater)
                                                                    except:
                                                                        print("locater can't be assigned")
                                                                break
                                                        quotaleft = "available arm quota remaining: " + str(quota) + "in extensive loop" + '\n'
                                                        compno +=1
                                                        compcount = "[enumeration no:]" + str(compno)
                                                        complog.append(str(compcount) + "\n")
                                                        #add core information after finding correct index of epitope
                                                        complog.insert(0, armdeduce[0])
                                                        finalresults.append(complog)
                                                        if debug:
                                                            debug_logs.append(str(compcount) + "\n")
                                                            debug_logs.append(str(quotaleft) + "\n")
                                                            debug_logs.append("now the complog is" + "\n")
                                                            debug_logs.append(str(complog) + "\n")
                                                        complog = [] #reset the complog
                                                        break #leave the for i loop?   
                                                    else:
                                                        continue
                                                else:
                                                    extbreak.append("False")
                                                    """
                                    if quota == 0:
                                        #if the composition reaches zero, export the complog and reset to [] for next enumeration
                                        #if no, try to add extensive epitopes until no units can be added OR it reaches zero
                                        if debug:
                                            debug_logs.append("quota is 0, checking if the composition gets zero" + "\n")
                                        #check avgunit and break in case previous break didnt work
                                        if testzero(avgunit):
                                            #find locator part
                                            #get the location of "first - largest epitope" when the value reaches zero
                                            #correct iint should be the index of the first epitope being added
                                            #ll = locater + 1
                                            for kkkkk in verb8_groupAepitopes[locater:]:
                                                #debug
                                                #print(f"[debug] comparing epitope is {kkkkk[0]} while current latest comp is {complog[0]}")
                                                if str(kkkkk[0]) == str(complog[0]):
                                                    iint = verb8_groupAepitopes.index(kkkkk)
                                                    print(f"[dev]: indx of {kkkkk[0]} is {type(iint)} {iint}")
                                                    skipiter = "[dev]: indx of " + str(kkkkk[0]) + " is " + str(type(iint)) + str(iint) + '\n'
                                                    #dev, to tell
                                                    if locater >= iint:
                                                        pass
                                                    else:
                                                        try:
                                                            locater = int(iint) +1
                                                            #locater = int(iint) +1 #start from next we had found
                                                            resetlocater = "[dev] reset locater to " + str(locater) + '\n'
                                                            if debug:
                                                                debug_logs.append(skipiter)
                                                                debug_logs.append(resetlocater)
                                                            print(resetlocater)
                                                        except:
                                                            print("locater can't be assigned")
                                                    break
                                            quotaleft = "available arm quota remaining: " + str(quota) + '\n'
                                            compno +=1
                                            compcount = "[enumeration no:]" + str(compno)
                                            complog.append(str(compcount) + "\n")
                                            #add core information after finding correct index of epitope
                                            complog.insert(0, armdeduce[0])
                                            finalresults.append(complog)
                                            if debug:
                                                debug_logs.append(str(compcount) + "\n")
                                                debug_logs.append(str(quotaleft) + "\n")
                                                debug_logs.append("BREAKING outside from epitope check, please check" + "\n")
                                                debug_logs.append(str(complog) + "\n")
                                            complog = [] #reset the complog
                                        else:
                                            #try to add extensive epitopes until no units can be added OR it reaches zero
                                            ########################

                                            quo0counter = 0
                                            quo0list = [] #define it
                                            while not(quo0counter >9 or len(quo0list)==len(verb8_groupBeptiopes)):
                                                quo0list = [] #init and clean it in the beginning
                                                quo0counter+=1
                                                for j in verb8_groupBeptiopes:
                                                #add extensive epitopes if the composition is not zero
                                                    aa, bb, finalvaldict = compactcompcalc(avgunit, j, 3)
                                                    if debug:
                                                        extendb = "Looping extensive from " + str(j[0]) + ", avg unit = " + str(avgunit) + " while the quota is zero"+ '\n'
                                                        debug_logs.append(extendb)
                                                        debug_logs.append("quota left is " + str(quota) + "\n")
                                                    if aa:
                                                        #cc = "".join(bb)
                                                        avgunit.update(finalvaldict)
                                                        print(f"[dev]avgunit get updated to {avgunit} in extensive loop")
                                                        #quota-=1
                                                        complog.append(j[0]) #if the error happens, change to str(i[0])
                                                        if testzero(finalvaldict):#when the composition reaches zero
                                                            compno +=1
                                                            compcount = "[enumeration no:]" + str(compno)
                                                            complog.append(str(compcount) + "\n")
                                                            #add core information after finding correct index of epitope
                                                            complog.insert(0, armdeduce[0])
                                                            finalresults.append(complog)
                                                            quotaleft = "available arm quota remaining: " + str(quota) + '\n'
                                                            if debug:
                                                                debug_logs.append(str(compcount) + "\n")
                                                                debug_logs.append(str(quotaleft) + "\n")
                                                                debug_logs.append("now the complog is" + "\n")
                                                                debug_logs.append(str(complog) + "\n")
                                                            complog = [] #reset the complog
                                                            break #leave the for i loop?   
                                                    else:
                                                        quo0list.append("False")
                                                        continue
                                                quo0counter+=1 #break when it reach 10
                                            #the composition cannot be satisfied.
                                            complog = []
                                            if debug:
                                                debug_logs.append("End enumeration when the composition cannot be satisfied" + "\n")
                                    else:
                                        print("quota nested loop exception")
                                        if debug:
                                            debug_logs.append("quota nested loop exception" + "\n")
                                        break
                                    #avgunit calculation indent
                                else:
                                    if debug:
                                        debug_logs.append("avgunit isn't able to do enumeration, break it" + "\n")
                                    break
                            debug_logs.append("armsforepitopes -1, go to the next loop" + "\n")
                        #while armsforepitope indent?
                            #code
                                    #check if locater reach the limit
                                    #before the limit, each time reaches zero will only change locater, not reducing armsno
                                    #armsnoepitope-1? 
                                #do remaining residue test
                                #do remaining residue test
                            print(f"enumeration on temp armsno = {armsforepitopes} finished") #debug print
                            armsforepitopes -=1
                            #dont get why while armsforepitopes > 0  an get zero division error
                            if armsforepitopes == 0:
                                print("armsforepitopes reaches 0, exit loop")
                                break
                        print("after all calculation in this arm no, arms-1")          
                        armsno -= 1
                        #print(f"Current possible composition:{complog}")
                else:
                    invalidstruc = "No composition possible for current arms" + str(armsno)
                    print(invalidstruc)
                    if debug:
                        debug_logs.append(invalidstruc + "\n")
                    armsno -= 1 
                    #reduce one arm until it gets 0 and break the loop
            else:
                print("error when doing tricks on armsno...")
    #print(f"[dev]complogs are {complog}") 
    #print(f"[dev]finalresults are {finalresults}")
    if debug:
        debug_logs.append("Finished deducing glycan composition. Summary: \n")
        debug_logs.append("Check if the complog get clean in previous steps \n")
        debug_logs.append(str(complog) + "\n")
        debug_logs.append("Finished deducing. Final results are: \n")
        debug_logs.append(str(finalresults) + "\n")
        #open file, write timestamp first so we can STACK them in same file
        with open("enumlogs_v8b.txt", "a") as debuglogs:
            for lines in debug_logs:
                debuglogs.writelines(lines)
        print("Finished writing debug logs...")
    if (len(finalresults) == 1 and finalresults[0] == []):
        finalresults.pop(0)
        return "No candidate composition found" #change to False in future for further integration within code blocks
    return finalresults


    
def deducecompositiona(specific_epitopes, speccompositions,debug=False):
    print("developing, now testing without enzyme and fragments")
    #create index first
    index = 0
    #print(f"len of input compositions: {len(speccompositions)}")  #this calculated all strs so it will give like 6~10 len of each comp
    #print("Input pd.Series")
    strlist1 = speccompositions["predictedcomp"]
    list2 = ast.literal_eval(strlist1)
    #add log container
    jjlog = {}
    for j in range(len(list2)):
        #print(type(list2[j][1]))
        #print(list2[j][1])
        complist = ast.literal_eval(list2[j][1]) #convert string to tuple
        spectrapredcomp = predcomplsttodict(complist) #convert tuple to dict (add KDN if structure has "6" components)
        #this extracted tuple value is what we're going to calculate...or manipulate? in real.
        #deducer(specific_epitopes, spectrapredcomp)
        if debug:
            jjlog[j] = deducer(demo_epitopes, spectrapredcomp,debug=True)
        else:
            jjlog[j] = deducer(demo_epitopes, spectrapredcomp)  #using demo_epitopes, not the specific epitopes
    return jjlog




In [141]:

#calculate time spend
start = time.time()
#read 
sample1=pd.read_csv('Annotated_revised_zfNGintenstine_20230612_cloud.csv', sep='\t')
readtime = abs(start - time.time())
print(f"reading csv took {readtime} seconds.")

#edit the value to read certain rown from input full converted spec csv  
spec_entry = 23
#no error-detecting function rn

speccompositions = sample1.head(1) #24 for one spectrum has multiple assignments
tmpseries = sample1.iloc[spec_entry]
#print(type(sample1.iloc[2])) #series
strlist123 = tmpseries["predictedcomp"]
print(f"the input compositions at row {spec_entry} is {strlist123}")
#print(tmpseries)
#deducecompositiona(deduceinput, speccompositions)  #for df
final_output = deducecompositiona(deduceinput, tmpseries,debug=True)
print(f"the enumeration output is {final_output}")
#print(type(speccompositions))  #it's pandas df
timer1 = time.time()-start
timer2 = time.time()
print('time spent for strucutre prediction on loaded csv and glycotope info: ', timer1, 'seconds.')
#https://stackoverflow.com/questions/36459969/how-to-convert-a-list-to-a-dictionary-with-indexes-as-values isn't working

#second prompt
tmpseries1 = sample1.iloc[2]
final_output2 = deducecompositiona(deduceinput, tmpseries1)
print(f"the enumeration output is {final_output2}")
timer3 = time.time()-start #from beginning
timer4 = time.time()-timer2 #diff from previous
print('time spent for strucutre prediction on loaded csv and glycotope info: ', timer3, 'seconds.')
print('delta time from previous analysis is',  timer4, 'seconds more.')



reading csv took 0.017539024353027344 seconds.
the input compositions at row 23 is [(1843.954, '(0, 4, 4, 0, 0, 0)'), (1845.945, '(0, 3, 2, 1, 0, 1)')]
developing, now testing without enzyme and fragments
predicted composition dict is {'F': 0, 'H': 4, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
[debug] 2023-12-14 15:58:13 Mode on and logs has been initiated: []
[debug] input arms number -1, original comp{'F': 0, 'H': 4, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}, Composition for calculation (-H3N2) {'F': 0, 'H': 1, 'N': 2, 'S': 0, 'G': 0, 'KDN': 0}
Start corrupted NG searching
[epitope added]: LacNAc attached on core
[epitope added]: GlcNAc attached on core
Finished writing debug logs...
predicted composition dict is {'F': 0, 'H': 3, 'N': 2, 'S': 1, 'G': 0, 'KDN': 1}
[debug] 2023-12-14 15:58:13 Mode on and logs has been initiated: []
[debug] input arms number -1, original comp{'F': 0, 'H': 3, 'N': 2, 'S': 1, 'G': 0, 'KDN': 1}, Composition for calculation (-H3N2) {'F': 0, 'H': 0, 'N': 0, 'S': 1, 'G': 0, '

In [142]:
#calculations only works on dict, not list or tuple... meaning we worked in vain on making them list
#https://stackoverflow.com/questions/17671875/how-to-subtract-values-from-dictionaries
adict = {"F":1, "H":5 , "N": 4, "S":1}
bdict = {"H":3, "N":2}
cdict = {key: adict[key] - bdict.get(key, 0) for key in adict}
print(cdict)

{'F': 1, 'H': 2, 'N': 2, 'S': 1}


In [143]:
art = tmpseries1["predictedcomp"]
print(art)
#print(tmpseries1)
#artificail input
artseries1 = pd.Series(["[(2048.053, '(0, 5, 4, 0, 0, 0)')]", 1], index=["predictedcomp", "MS1scan no"]) #bianten core test
artseries2 = pd.Series(["[(2583.316, '(1, 5, 4, 1, 0)')]", 556], index=["predictedcomp", "MS1scan no"]) #bianten core test
#print(artseries1["predictedcomp"])
#strlist1 = artseries1["predictedcomp"]
#list2 = ast.literal_eval(strlist1)
#print(list2)

[(1639.854, '(0, 3, 4, 0, 0, 0)')]


In [144]:
deducecompositiona(deduceinput, artseries1,debug=True)


developing, now testing without enzyme and fragments
predicted composition dict is {'F': 0, 'H': 5, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
[debug] 2023-12-14 15:58:13 Mode on and logs has been initiated: []
now possible arms is biante
[debug] input arms number 2, original comp{'F': 0, 'H': 5, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}, Composition for calculation (-H3N2) {'F': 0, 'H': 2, 'N': 2, 'S': 0, 'G': 0, 'KDN': 0}
arms no: 2, enumerates on 4
arms no: 2, enumerates on 3
arms no: 2, enumerates on 2
2arms enumerating and now the valdict is {'F': 0, 'H': 0, 'N': 0, 'S': 0, 'G': 0, 'KDN': 0} with enumerating on Bi-antennary N-glycan core
Reaches 0 when only fitting basic structure for NG arms Bi-antennary N-glycan core
arms no: 1, enumerates on 2
arms no: 1, enumerates on 1
1arms enumerating and now the valdict is {'F': 0, 'H': -5, 'N': -1, 'S': 0, 'G': 0, 'KDN': 0} with enumerating on Bi-antennary hybrid N-glycan core
No composition possible for current arms1
Finished writing debug logs...


{0: [['[enumeration no:]1', 'Bi-antennary N-glycan core', '0']]}

In [145]:
deducecompositiona(deduceinput, artseries2,debug=True)

developing, now testing without enzyme and fragments
predicted composition dict is {'F': 1, 'H': 5, 'N': 4, 'S': 1, 'G': 0}
[debug] 2023-12-14 15:58:13 Mode on and logs has been initiated: []
now possible arms is biante
[debug] input arms number 2, original comp{'F': 1, 'H': 5, 'N': 4, 'S': 1, 'G': 0}, Composition for calculation (-H3N2) {'F': 1, 'H': 2, 'N': 2, 'S': 1, 'G': 0}
arms no: 2, enumerates on 4
arms no: 2, enumerates on 3
arms no: 2, enumerates on 2
2arms enumerating and now the valdict is {'F': 1, 'H': 0, 'N': 0, 'S': 1, 'G': 0} with enumerating on Bi-antennary N-glycan core
now the temp looping arms = 2. Start terminal enumeration.
[dev] Calculate avg units carrying on arms = 2. Real arms = 2
reset avg {}, comp {'F': 1, 'H': 2, 'N': 2, 'S': 1, 'G': 0} and local arms 2, global 2
rebuiling logic here
locater +1 when able to add. In next loop we don't need it, start from next one...
enumeration on temp armsno = 2 finished
reset avg {}, comp {'F': 1, 'H': 2, 'N': 2, 'S': 1, 'G

{0: [['Bi-antennary N-glycan core',
   'Sialyl-Lewis AX antigen',
   'LacNAc',
   '[enumeration no:]1\n']]}

In [146]:
#extra large
artseries3 = pd.Series(["[(4565.289, '(1, 7, 6, 4, 0, 0)')]", 123456], index=["predictedcomp", "MS1scan no"]) #bianten core test
deducecompositiona(deduceinput, artseries3,debug=True)


developing, now testing without enzyme and fragments
predicted composition dict is {'F': 1, 'H': 7, 'N': 6, 'S': 4, 'G': 0, 'KDN': 0}
[debug] 2023-12-14 15:58:13 Mode on and logs has been initiated: []
now possible arms is biante
now possible arms is triante
now possible arms is tetraante
now possible arms is bisect
[debug] input arms number 4, original comp{'F': 1, 'H': 7, 'N': 6, 'S': 4, 'G': 0, 'KDN': 0}, Composition for calculation (-H3N2) {'F': 1, 'H': 4, 'N': 4, 'S': 4, 'G': 0, 'KDN': 0}
arms no: 4, enumerates on 4
4arms enumerating and now the valdict is {'F': 1, 'H': 0, 'N': 0, 'S': 4, 'G': 0, 'KDN': 0} with enumerating on Tetra-antennary N-glycan core
now the temp looping arms = 4. Start terminal enumeration.
[dev] Calculate avg units carrying on arms = 4. Real arms = 4
reset avg {}, comp {'F': 1, 'H': 4, 'N': 4, 'S': 4, 'G': 0, 'KDN': 0} and local arms 4, global 4
rebuiling logic here
locater +1 when able to add. In next loop we don't need it, start from next one...
enumerati

{0: []}